# Person 1 Contribution: LoRA Fine-Tuning Extension

**Contributor:** Eram Mahamud  
Student ID: 48982539

**Role:** Person 1  
**Section:** P3 LoRA Extension + P5 Integration  
**Base model:** `Qwen/Qwen2.5-1.5B-Instruct` used as a practical Colab fallback because Llama 3 requires gated Hugging Face approval.

This notebook adds a LoRA-based customer service response module to the existing group project. It is designed to support the P3 response-generation section and provide a clean integration function for P5.

## Important clarification for marking

P1's responsibility is to **add the LoRA implementation and integration code** as an individual contribution. Running the full training or final deployment can be handled by the integration member/P5 or whoever has the required Hugging Face access and GPU resources.

## What this notebook provides

1. LoRA training/adaptation section using the tutor-approved Bitext dataset and a Colab-runnable Qwen fallback model. the approved Bitext Customer Support Dataset.
2. Adapter saving path: `./lora-weights`.
3. Corrected P3 replacement for `call_ollama()`.
4. P5 integration functions to load and use the saved LoRA adapter.
5. Report paragraph and Git commit note for Eram's contribution.

## Note about Llama 3 access

This notebook intentionally uses Llama 3, not a smaller substitute model, because the group P3 section is based on Llama 3. To actually run it, the runtime must have Hugging Face access to the model and enough GPU memory. If P5 is only integrating the work, they need the `./lora-weights` folder created by the LoRA training step.

## 1. Install Required Libraries

Run this cell once. If you are using Google Colab, restart the runtime if installation asks for it.

In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


## 2. Import Libraries

In [2]:
import os
import torch
import pandas as pd
from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Configuration

The original project direction mentioned Llama 3, but Meta Llama models require gated Hugging Face approval and may not run immediately in Colab. For a runnable assignment demonstration, this notebook uses **Qwen/Qwen2.5-1.5B-Instruct** as the practical fallback base model and applies LoRA fine-tuning on top of it.

The important contribution is the **LoRA implementation and integration pipeline**: dataset formatting, LoRA adapter training, saving `./lora-weights`, P3 `call_ollama()` replacement, P5 integration, and evaluation.

In [3]:
# ============================================================
# Configuration
# ============================================================

# Llama 3 base model used for the group P3 response-generation section.
# Running this requires Hugging Face access and sufficient GPU memory.

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

# This folder is what P3/P5 will use later
LORA_OUTPUT_DIR = "./lora-weights"

MAX_LENGTH = 512
TRAIN_SAMPLE_SIZE = 2000      # P5/integration person can reduce/increase based on GPU/time
EVAL_SAMPLE_SIZE = 300

EPOCHS = 1                    # Increase to 2 or 3 if time/GPU allows
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4

SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Base model:", BASE_MODEL_NAME)
print("LoRA output path:", LORA_OUTPUT_DIR)

Using device: cuda
Base model: Qwen/Qwen2.5-1.5B-Instruct
LoRA output path: ./lora-weights


## 4. Load the Approved Bitext Customer Support Dataset

In [4]:
dataset = load_dataset(DATASET_NAME)

print(dataset)
print('Columns:', dataset['train'].column_names)

df = dataset['train'].to_pandas()
print('Dataset shape:', df.shape)
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})
Columns: ['flags', 'instruction', 'category', 'intent', 'response']
Dataset shape: (26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


## 5. Detect Dataset Columns

This makes the notebook more robust if the dataset column names are slightly different.

In [5]:
def detect_dataset_columns(dataframe):
    columns = dataframe.columns.tolist()

    instruction_col = None
    response_col = None
    category_col = None
    intent_col = None

    instruction_candidates = [
        "instruction", "input", "query", "customer_query", "text", "utterance"
    ]
    response_candidates = [
        "response", "output", "answer", "reply"
    ]

    for col in columns:
        lower_col = col.lower().strip()

        if lower_col in instruction_candidates:
            instruction_col = col

        if lower_col in response_candidates:
            response_col = col

        if lower_col == "category":
            category_col = col

        if lower_col == "intent":
            intent_col = col

    if instruction_col is None:
        raise ValueError(f"Instruction/customer query column not found. Available columns: {columns}")

    if response_col is None:
        raise ValueError(f"Response/answer column not found. Available columns: {columns}")

    return instruction_col, response_col, category_col, intent_col


instruction_col, response_col, category_col, intent_col = detect_dataset_columns(df)

print('Instruction column:', instruction_col)
print('Response column:', response_col)
print('Category column:', category_col)
print('Intent column:', intent_col)

Instruction column: instruction
Response column: response
Category column: category
Intent column: intent


## 6. Create LoRA Training Prompts

In [6]:
def create_lora_prompt(row):
    customer_message = str(row[instruction_col])
    response = str(row[response_col])

    category_text = ""
    intent_text = ""

    if category_col is not None:
        category_text = f"Category: {row[category_col]}\n"

    if intent_col is not None:
        intent_text = f"Intent: {row[intent_col]}\n"

    prompt = f"""### Task:
You are a professional and helpful customer service assistant.
Your job is to respond clearly, politely, and appropriately to the customer's message.

{category_text}{intent_text}### Customer Message:
{customer_message}

### Customer Service Response:
{response}"""

    return prompt


df['lora_text'] = df.apply(create_lora_prompt, axis=1)

print(df['lora_text'].iloc[0])

### Task:
You are a professional and helpful customer service assistant.
Your job is to respond clearly, politely, and appropriately to the customer's message.

Category: ORDER
Intent: cancel_order
### Customer Message:
question about cancelling order {{Order Number}}

### Customer Service Response:
I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


## 7. Create Train and Evaluation Splits

In [7]:
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

max_required = TRAIN_SAMPLE_SIZE + EVAL_SAMPLE_SIZE
if len(df) < max_required:
    print(f"Warning: dataset has only {len(df)} rows. Using available rows.")
    TRAIN_SAMPLE_SIZE = int(len(df) * 0.85)
    EVAL_SAMPLE_SIZE = len(df) - TRAIN_SAMPLE_SIZE

train_df = df.iloc[:TRAIN_SAMPLE_SIZE].copy()
eval_df = df.iloc[TRAIN_SAMPLE_SIZE:TRAIN_SAMPLE_SIZE + EVAL_SAMPLE_SIZE].copy()

print('LoRA train size:', len(train_df))
print('LoRA eval size:', len(eval_df))

train_dataset = Dataset.from_pandas(train_df[['lora_text']])
eval_dataset = Dataset.from_pandas(eval_df[['lora_text']])

LoRA train size: 2000
LoRA eval size: 300


## 8. Load Tokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = 'right'
print('Tokenizer loaded successfully.')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully.


## 9. Tokenize Dataset

In [9]:
def tokenize_lora_data(example):
    tokenized = tokenizer(
        example['lora_text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH
    )

    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized


tokenized_train = train_dataset.map(
    tokenize_lora_data,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_eval = eval_dataset.map(
    tokenize_lora_data,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print(tokenized_train)
print(tokenized_eval)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 300
})


## 10. Load Base Model

This cell first tries 4-bit loading. If that fails, it loads the model normally.

In [10]:
try:
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4'
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto'
    )

    model = prepare_model_for_kbit_training(model)
    print('Model loaded using 4-bit quantization.')

except Exception as e:
    print('4-bit loading failed. Loading standard model instead.')
    print('Reason:', e)

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto' if torch.cuda.is_available() else None
    )

model.config.use_cache = False
print('Base model loaded.')

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded using 4-bit quantization.
Base model loaded.


## 11. Add LoRA Configuration

In [11]:
# These target modules work for Llama-style architectures, including Llama 3.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)

print('Trainable parameters after applying LoRA:')
model.print_trainable_parameters()

Trainable parameters after applying LoRA:
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


## 12. Train LoRA Adapter

In [12]:
training_args = TrainingArguments(
    output_dir='./eram_lora_training_logs',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    remove_unused_columns=False
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    # tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,0.848973,0.827296
200,0.745240,0.756019


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=250, training_loss=0.8734701099395752, metrics={'train_runtime': 1161.4768, 'train_samples_per_second': 1.722, 'train_steps_per_second': 0.215, 'total_flos': 8077509132288000.0, 'train_loss': 0.8734701099395752, 'epoch': 1.0})

## 13. Save LoRA Adapter to `./lora-weights`

This folder is what P3 and P5 need.

In [13]:
os.makedirs(LORA_OUTPUT_DIR, exist_ok=True)

model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print('Eram LoRA adapter saved at:', LORA_OUTPUT_DIR)
print('Files:', os.listdir(LORA_OUTPUT_DIR))

Eram LoRA adapter saved at: ./lora-weights
Files: ['README.md', 'tokenizer.json', 'chat_template.jinja', 'adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json']


## 14. Test the LoRA Customer Service Response

In [14]:
def generate_lora_customer_response(
    customer_message,
    category=None,
    sentiment=None,
    intent=None,
    max_new_tokens=180
):
    category_text = f"Predicted Category: {category}\n" if category else ""
    sentiment_text = f"Predicted Sentiment: {sentiment}\n" if sentiment else ""
    intent_text = f"Predicted Intent: {intent}\n" if intent else ""

    prompt = f"""### Task:
You are a professional customer service assistant.
Use the available information and respond politely, clearly, and helpfully.

{category_text}{sentiment_text}{intent_text}### Customer Message:
{customer_message}

### Customer Service Response:
"""

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)

    if torch.cuda.is_available():
        inputs = {key: value.to(model.device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if '### Customer Service Response:' in decoded:
        return decoded.split('### Customer Service Response:')[-1].strip()

    return decoded.strip()


test_message = 'I received the wrong item and I want a refund.'

test_response = generate_lora_customer_response(
    customer_message=test_message,
    category='order issue',
    sentiment='negative',
    intent='refund request'
)

print('Customer Message:')
print(test_message)
print('LoRA Customer Service Response:')
print(test_response)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_s

Customer Message:
I received the wrong item and I want a refund.
LoRA Customer Service Response:
We Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task Task


# 15. Evaluation of Eram's LoRA Adapter

This section evaluates the LoRA adapter after training. It includes:

1. **Trainer evaluation loss** on the validation split.
2. **Perplexity**, which gives a language-model quality indicator.
3. **Qualitative response evaluation** on sample customer queries.
4. **Simple lexical overlap score** between generated and reference responses.
5. A saved CSV file containing evaluation examples for the report or P5 integration evidence.

Run these cells after the training, saving, and `generate_lora_customer_response()` function cells have completed.

In [15]:
# ============================================================
# 15.1 Quantitative Evaluation: Eval Loss and Perplexity
# ============================================================

import math

# Evaluate the trained LoRA model on the evaluation set
lora_eval_metrics = trainer.evaluate(eval_dataset=tokenized_eval)

print("=" * 70)
print(" LoRA Evaluation Metrics")
print("=" * 70)
for key, value in lora_eval_metrics.items():
    print(f"{key}: {value}")

# Perplexity is commonly used for language-model evaluation.
# Lower perplexity generally indicates better next-token prediction.
if "eval_loss" in lora_eval_metrics:
    try:
        perplexity = math.exp(lora_eval_metrics["eval_loss"])
    except OverflowError:
        perplexity = float("inf")
    print(f"Perplexity: {perplexity:.4f}")
else:
    perplexity = None
    print("eval_loss was not found, so perplexity could not be calculated.")

 LoRA Evaluation Metrics
eval_loss: 0.7469896674156189
eval_runtime: 54.5069
eval_samples_per_second: 5.504
eval_steps_per_second: 2.752
epoch: 1.0
Perplexity: 2.1106


In [16]:
# ============================================================
# 15.2 Helper Functions for Response-Level Evaluation
# ============================================================

import re
import pandas as pd


def normalize_text(text):
    """Lowercase and clean text for simple lexical comparison."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def word_overlap_score(reference, generated):
    """
    Simple Jaccard-style word overlap between reference and generated response.
    This is not a perfect semantic metric, but it is useful as a lightweight
    assignment-level sanity check.
    """
    ref_words = set(normalize_text(reference).split())
    gen_words = set(normalize_text(generated).split())

    if len(ref_words) == 0 and len(gen_words) == 0:
        return 1.0
    if len(ref_words) == 0 or len(gen_words) == 0:
        return 0.0

    return len(ref_words.intersection(gen_words)) / len(ref_words.union(gen_words))


def response_word_count(text):
    return len(normalize_text(text).split())

In [17]:
# ============================================================
# 15.3 Generate Responses on Evaluation Samples
# ============================================================

# Keep this small so it runs quickly in Colab.
NUM_EVAL_GENERATION_SAMPLES = 20

sample_eval_df = eval_df.head(NUM_EVAL_GENERATION_SAMPLES).copy()

evaluation_rows = []

for idx, row in sample_eval_df.iterrows():
    customer_message = str(row[instruction_col])
    reference_response = str(row[response_col])
    category = str(row[category_col]) if category_col is not None else None
    intent = str(row[intent_col]) if intent_col is not None else None

    generated_response = generate_lora_customer_response(
        customer_message=customer_message,
        category=category,
        sentiment=None,
        intent=intent,
        max_new_tokens=160
    )

    overlap = word_overlap_score(reference_response, generated_response)

    evaluation_rows.append({
        "sample_id": idx,
        "customer_message": customer_message,
        "category": category,
        "intent": intent,
        "reference_response": reference_response,
        "generated_lora_response": generated_response,
        "reference_word_count": response_word_count(reference_response),
        "generated_word_count": response_word_count(generated_response),
        "word_overlap_score": overlap
    })

lora_response_eval_df = pd.DataFrame(evaluation_rows)

print("Generated evaluation samples:")
display(lora_response_eval_df[[
    "sample_id",
    "customer_message",
    "reference_response",
    "generated_lora_response",
    "word_overlap_score"
]])

Generated evaluation samples:


,sample_id,customer_message,reference_response,generated_lora_response,word_overlap_score
0,2000,"I have lost my account access key, I need help...","Oh no, I'm sorry to hear that you've lost your...",Thank you for reaching out! We understand that...,0.220859
1,2001,how could I use the {{Account Type}} profile?,Definitely! I'm here to assist you with utiliz...,We're here to provide you with guidance on how...,0.273333
2,2002,I don't know how to see how soon can I expect ...,We understand that tracking the estimated arri...,Absolutely! To assist you in understanding how...,0.226415
3,2003,could you help me enter the secondary delivery...,I fathom that you're seeking assistance in ent...,Of course! I'm here to assist you with enterin...,0.400000
4,2004,I do not know how I can enter a different ship...,"I'm on the same page, your confusion about ent...",Absolutely! It's great that you're unsure abou...,0.343511
5,2005,can you help me take a quick look at my invoic...,I've understood that you're seeking assistance...,I'm glad to assist you in checking your invoic...,0.264957
6,2006,i try to check the early exit penalties,Assuredly! I can assist you in checking the ea...,I'm aware that you're trying to check the earl...,0.200000
7,2007,i need assistance to check how long the shgipm...,We understand that you are eager to know the e...,We're here to help! I understand that you woul...,0.270000
8,2008,I do not know what to do to set my shipping ad...,I appreciate that you're unsure about the step...,Thank you for reaching out! I understand that ...,0.294118
9,2009,I want to inform of problems with signup,Thank you for reaching out to us regarding the...,We understand that you're facing difficulties ...,0.371429


In [18]:
# ============================================================
# 15.4 Evaluation Summary
# ============================================================

mean_overlap = lora_response_eval_df["word_overlap_score"].mean()
mean_reference_len = lora_response_eval_df["reference_word_count"].mean()
mean_generated_len = lora_response_eval_df["generated_word_count"].mean()

print("=" * 70)
print(" LoRA Response-Level Evaluation Summary")
print("=" * 70)

print(f"Number of generated evaluation samples: {len(lora_response_eval_df)}")
print(f"Average word-overlap score: {mean_overlap:.4f}")
print(f"Average reference response length: {mean_reference_len:.2f} words")
print(f"Average generated response length: {mean_generated_len:.2f} words")

if perplexity is not None:
    print(f"Perplexity from validation loss: {perplexity:.4f}")

print("\nInterpretation:")
print("- Eval loss and perplexity evaluate token prediction quality.")
print("- Word overlap is a lightweight comparison with the dataset response.")
print("- Generated examples should also be manually checked for tone, clarity, relevance, and politeness.")

 LoRA Response-Level Evaluation Summary
Number of generated evaluation samples: 20
Average word-overlap score: 0.2966
Average reference response length: 113.50 words
Average generated response length: 117.80 words
Perplexity from validation loss: 2.1106

Interpretation:
- Eval loss and perplexity evaluate token prediction quality.
- Word overlap is a lightweight comparison with the dataset response.
- Generated examples should also be manually checked for tone, clarity, relevance, and politeness.


In [19]:
# ============================================================
# 15.5 Save Evaluation Results for Report/P5 Evidence
# ============================================================

EVALUATION_OUTPUT_CSV = "./eram_lora_evaluation_results.csv"

lora_response_eval_df.to_csv(EVALUATION_OUTPUT_CSV, index=False)

print("Evaluation results saved to:", EVALUATION_OUTPUT_CSV)
print("This CSV can be included as evidence of Person 1 LoRA evaluation.")

Evaluation results saved to: ./eram_lora_evaluation_results.csv
This CSV can be included as evidence of Person 1 LoRA evaluation.


## Evaluation Note for the Report

The LoRA extension was evaluated using validation loss, perplexity, and sample response generation. Validation loss and perplexity provide quantitative language-model indicators, while the generated examples allow manual inspection of customer-service quality, politeness, relevance, and clarity. A lightweight word-overlap score was also calculated against the dataset reference response to provide a simple comparison measure. Because customer-service responses can be valid even when phrased differently from the reference answer, the qualitative examples should be considered together with the numerical metrics.

# P3 Replacement Section

After running the training section above, the folder `./lora-weights` will exist.

Now in the P3 notebook, find the old cell that contains:

```python
def call_ollama(...)
```

Replace that entire old cell with the cell below.

This keeps the rest of P3 unchanged, including prompt functions and `final_response_for_UI`.

In [20]:
!pip install torchao --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.0 MB/s eta 0:00:00


In [21]:
# ============================================================
# Person 1 Contribution: Eram Mahamud
# LoRA-based replacement for call_ollama() in P3
# ============================================================

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

LORA_WEIGHTS_PATH = './lora-weights'

# Use the same base model used during LoRA training.
# This must match the base model used during LoRA training.
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LORA_WEIGHTS_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None
)

lora_model = PeftModel.from_pretrained(
    base_model,
    LORA_WEIGHTS_PATH
)

lora_model.eval()

def call_ollama(prompt, model=lora_model, temperature=0.3, num_predict=250):
    """
    Replaces the original Ollama call with the LoRA-adapted local model.
    The rest of the P3 notebook can stay exactly the same.
    """

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=2048
    )

    if torch.cuda.is_available():
        inputs = {key: value.to(model.device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=num_predict,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if response.startswith(prompt):
        response = response[len(prompt):].strip()

    return response

print('LoRA call_ollama() replacement loaded successfully.')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LoRA call_ollama() replacement loaded successfully.


# P5 Integration Section

P5 can use the following cells to load Eram's saved LoRA adapter and connect it with the final UI pipeline.

In [22]:
# ============================================================
# P5 Integration: Load Eram's LoRA Adapter
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_NAME_FOR_P5 = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_ADAPTER_PATH_FOR_P5 = './lora-weights'

def load_eram_lora_model():
    tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH_FOR_P5)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME_FOR_P5,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto' if torch.cuda.is_available() else None
    )

    lora_model = PeftModel.from_pretrained(
        base_model,
        LORA_ADAPTER_PATH_FOR_P5
    )

    lora_model.eval()
    return tokenizer, lora_model


def eram_lora_response_for_p5(
    customer_message,
    tokenizer,
    model,
    category=None,
    sentiment=None,
    intent=None,
    max_new_tokens=180
):
    category_text = f"Predicted Category: {category}\n" if category else ""
    sentiment_text = f"Predicted Sentiment: {sentiment}\n" if sentiment else ""
    intent_text = f"Predicted Intent: {intent}\n" if intent else ""

    prompt = f"""### Task:
You are a professional customer service assistant.
Use the analysis information below and write a helpful response.

{category_text}{sentiment_text}{intent_text}### Customer Message:
{customer_message}

### Customer Service Response:
"""

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)

    if torch.cuda.is_available():
        inputs = {key: value.to(model.device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if '### Customer Service Response:' in decoded:
        return decoded.split('### Customer Service Response:')[-1].strip()

    return decoded.strip()

## Optional: Combined Final Pipeline for P5

If P5 already has the old `analyze()` function from P2/P3, this function can combine its output with Eram's LoRA-generated response.

In [23]:
def final_customer_service_pipeline(
    customer_message,
    analyze_function=None,
    tokenizer=None,
    lora_model=None
):
    analysis_result = {}

    if analyze_function is not None:
        try:
            analysis_result = analyze_function(customer_message)
        except Exception as e:
            print('Existing analyze() function failed:', e)
            analysis_result = {}

    category = None
    sentiment = None
    intent = None

    if isinstance(analysis_result, dict):
        category = analysis_result.get('category', None)
        sentiment = analysis_result.get('sentiment', None)
        intent = analysis_result.get('intent', None)

    if tokenizer is not None and lora_model is not None:
        final_response = eram_lora_response_for_p5(
            customer_message=customer_message,
            tokenizer=tokenizer,
            model=lora_model,
            category=category,
            sentiment=sentiment,
            intent=intent
        )
    else:
        final_response = 'LoRA model is not loaded.'

    return {
        'customer_message': customer_message,
        'analysis_result': analysis_result,
        'eram_lora_response': final_response
    }

## P5 Test Example

In [24]:
# ============================================================
# Example P5 Usage
# Run this only after ./lora-weights has been created.
# ============================================================

tokenizer_p5, lora_model_p5 = load_eram_lora_model()

customer_query = "I ordered a product but it arrived damaged. I want a replacement."

result = final_customer_service_pipeline(
    customer_message=customer_query,
    analyze_function=None,  # P5 can replace this with the existing analyze() function
    tokenizer=tokenizer_p5,
    lora_model=lora_model_p5
)

print(result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'customer_message': 'I ordered a product but it arrived damaged. I want a replacement.', 'analysis_result': {}, 'eram_lora_response': "We apologize for any inconvenience you've experienced with your recently received product that was damaged during shipping. We understand how important it is to have a proper replacement for this item. To assist you further, could you please provide us with some details about the product you purchased? This will help us locate an equivalent replacement and ensure a smooth resolution to your issue. Rest assured, we'll take all necessary steps to replace the damaged item promptly. Thank you for bringing this matter to our attention. Your satisfaction is our top priority!"}


In [25]:
# Run this only after ./lora-weights has been created.

tokenizer_p5, lora_model_p5 = load_eram_lora_model()

customer_query = 'I ordered a product but it arrived damaged. I want a replacement.'

result = final_customer_service_pipeline(
     customer_message=customer_query,
     analyze_function=None,  # P5 can replace this with the existing analyze() function
     tokenizer=tokenizer_p5,
     lora_model=lora_model_p5
)

print(result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'customer_message': 'I ordered a product but it arrived damaged. I want a replacement.', 'analysis_result': {}, 'eram_lora_response': 'We apologize for any inconvenience caused by the damage to your order. We understand that receiving an item in poor condition is not acceptable, and we are committed to resolving this issue promptly. To ensure you receive a replacement, please follow these steps:\n\n1. **Contact Us**: Reach out to our customer support team using the provided contact information on our website or through live chat on our platform. You can also email us at [customer@example.com] with details about the damaged product, including its serial number if available.\n\n2. **Provide Documentation**: During the conversation with our representatives, be sure to provide detailed photos of the damaged item along with any relevant packaging labels. This will help us accurately assess the extent of the damage and verify the authenticity of the complaint.\n\n3. **Request Replacement**:

In [26]:
import json

tokenizer_p5, lora_model_p5 = load_eram_lora_model()

customer_query = 'I ordered a product but it arrived damaged. I want a replacement.'

result = final_customer_service_pipeline(
    customer_message=customer_query,
    analyze_function=None,
    tokenizer=tokenizer_p5,
    lora_model=lora_model_p5
)

print("=" * 60)
print(" CUSTOMER MESSAGE")
print("=" * 60)
print(result['customer_message'])

print("\n" + "=" * 60)
print(" LORA MODEL RESPONSE")
print("=" * 60)
print(result['eram_lora_response'])

if result.get('analysis_result'):
    print("\n" + "=" * 60)
    print("📊  P2 ANALYSIS")
    print("=" * 60)
    for k, v in result['analysis_result'].items():
        print(f"  {k}: {v}")

print("\n" + "-" * 60)
print("Complete")
print("-" * 60)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

 CUSTOMER MESSAGE
I ordered a product but it arrived damaged. I want a replacement.

 LORA MODEL RESPONSE
We apologize for any inconvenience caused by your damaged order. We take responsibility to ensure that you receive the best possible product experience. To request a replacement, please contact our customer support team at <Customer Support Phone Number> or email us at <Customer Support Email>. They will be happy to assist you with the necessary steps to obtain a new item. Rest assured, we understand the importance of receiving a well-maintained product and appreciate your patience during this process. Thank you for bringing this matter to our attention. We value your satisfaction and look forward to resolving this issue promptly. How else can we help? #ProductSupport

### Analysis:
The message indicates dissatisfaction regarding the delivery of an ordered product which was found to be damaged upon arrival. The customer wants a replacement. This situation requires immediate attenti

# Report Paragraph for Eram's Contribution


**Person 1 Contribution: Eram Mahamud**

As Person 1, I contributed to the project by adding a LoRA-based parameter-efficient fine-tuning extension to the P3 customer service agent section. The existing system already used prompt engineering and language-model-based response generation, but LoRA was added to make the agent more specialised for customer service tasks. The original project direction considered Llama 3; however, because Meta Llama models require gated Hugging Face access, Qwen was used as a practical Colab-runnable fallback to demonstrate the same LoRA training and integration workflow. This extension uses the tutor-approved Bitext Customer Support Dataset to adapt the model toward customer support tone, context understanding, escalation handling, and response generation. The LoRA adapter is saved separately in `./lora-weights`, allowing P3 and P5 to load it without changing the existing preprocessing, classification, sentiment analysis, or UI pipeline. I also added an evaluation section using validation loss, perplexity, generated response examples, and a simple word-overlap comparison against reference responses.